# Vatican ebr. 530, Luke and John: PDF to OSIS

Converts Nehemia Gordon's transcription, translation and annotations of
*Biblioteca Apostolica Vaticana, Vat. ebr. 530*, part 1, fragment 11,
folios 1r-2v. The fragment carries **Luke 1:1-35** and **John 1:1-13**.

This is a thin driver over the tested `pdf2osis` package; the extraction lives
in `pdf2osis.glyphs`, `pdf2osis.layout` and `pdf2osis.ebr530`, so the same code
is covered by `tests/test_ebr530.py`.

## Why this PDF needs its own decoder

It is the mirror image of Sloane 237. There the ToUnicode CMap was broken and
the embedded font's `cmap` was the fix. Here the ToUnicode is sound but the
embedded subsets carry **no `cmap` table at all** — and `get_texttrace` merges
adjacent runs, sometimes two whole printed lines, into a single span, so
reversing a span swaps its runs.

So this profile reads `rawdict` and places every glyph by its own coordinates.
A combining mark is drawn at its base letter's left edge, which makes vowel
attachment exact rather than a nearest-neighbour guess. Character error on the
verified lines drops from ~16-26% to zero.

Run with the `tp` environment.

In [ ]:
from pathlib import Path

from pdf2osis import convert_pdf, get_profile

ROOT = Path.cwd().parent if Path.cwd().name == "python" else Path.cwd()
SOURCE = ROOT / "data" / "00_source_files"
OUTPUT = ROOT / "data" / "01_osis"

reports = {}
for key in ("ebr530_luke", "ebr530_john"):
    profile = get_profile(key)
    reports[key] = convert_pdf(profile.default_path(SOURCE), profile, OUTPUT)

for key, report in reports.items():
    print(f"{key}: {report.verses} verses, {report.note_definitions} footnotes")
    for variant, path in report.output_paths.items():
        print(f"    {variant:20s} {path.name}")

## Source anomalies

The translation column prints its own `(N)` verse markers, but they cannot be
trusted: page 2 prints `(3)` twice where the second should read `(4)`. Verses
are therefore anchored to the Hebrew column and the disagreement is reported.

In [ ]:
for key, report in reports.items():
    for anomaly in report.anomalies:
        print(f"{key}: {anomaly}")

## Non-verse manuscript text

The gospel heading becomes a `<title type="main">` inside a
`<div type="introduction">`, the `פרק ראשון` chapter heading a
`<title type="chapter">`, and each folio boundary a `<milestone type="pb">` at
its true position. Footnotes 1 and 2 annotate the gospel heading and footnote 6
the chapter heading, so none of them belongs to a verse at all.

In [ ]:
from lxml import etree

from pdf2osis.osis import OSIS_NS

NS = {"osis": OSIS_NS}
tree = etree.parse(str(reports["ebr530_luke"].output_paths["hebrew_commented"]))

heading = tree.xpath("//osis:div[@type='introduction']/osis:title", namespaces=NS)[0]
print("heading :", "".join(heading.itertext()))
print("chapter :", tree.xpath("//osis:title[@type='chapter']/text()", namespaces=NS)[0])
print("folios  :", tree.xpath("//osis:milestone[@type='pb']/@n", namespaces=NS))
print("notes   :", tree.xpath("//osis:note/@n", namespaces=NS))
print()


def verse_text(start):
    """Milestoned verse text runs from the sID element to the matching eID."""
    parts, node = [], start
    while (node := node.getnext()) is not None:
        if etree.QName(node).localname == "verse" and node.get("eID"):
            break
        parts.append(node.tail or "")
    return "".join([start.tail or ""] + parts).strip()


for verse in tree.xpath("//osis:verse[@sID][position() <= 2]", namespaces=NS):
    print(verse.get("osisID"), verse_text(verse)[:90])


## Output shape

Each book is written as three variants — plain Hebrew, annotated Hebrew, and
the English translation — with milestoned verses and one verse per line.

In [ ]:
path = reports["ebr530_john"].output_paths["hebrew"]
lines = path.read_text(encoding="utf-8").splitlines()
print(f"{path.name}: {len(lines)} lines, longest {max(len(line) for line in lines)} chars")
print()
for line in lines[-8:]:
    print(line)
